# earthkit-data: writing to targets

In this notebook you will see how to:

- write data into a target 
- use an encoder

First, we read some GRIB data from disk and load it into a fieldlist.

In [ ]:
import earthkit.data as ekd

d = ekd.from_source("file", "test4.grib")
fl = d.to_fieldlist()
len(fl)

### Targets

A target can represent a file, a database, a remote server etc. Data is written/added to a target by using a suitable encoder.
    
We use ``to_target()`` to write the data into target. The encoder is automatically guessed from the input data (when possible) or from the output path's suffix.

In [ ]:
# write all the data to a file
fl.to_target("file", "_my.grib")

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

In [ ]:
# write only a slice to target
fl[2:4].to_target("file", "_my.grib")

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

### Target as a context manager

A target can be used as a context manager.

In [ ]:
# write data to target
with ekd.create_target("file", "_my.grib") as t:
    t.write(fl)

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

The code below write each field individually to the target.

In [ ]:
# write data to target
with ekd.create_target("file", "_my.grib") as t:
    for f in fl:
        if f.parameter.variable() == "z":
            t.write(f)

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

### Using an encoder

In the example below we specify extra GRIB metadata. The GRIB encoder, which is used behind the scenes, will add this to the resulting file.

In [ ]:
# write to target and set GRIB keys for the output
fl[:2].to_target("file", "_my.grib", metadata={"date": 20250108, "bitsPerValue": 8})

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls(keys=["time.valid_datetime", "time.base_datetime", "time.step", "metadata.bitsPerValue"])

We can do the same by creating an encoder object.

In [ ]:
# create an encoder
encoder = ekd.create_encoder("grib", metadata={"date": 20250108})

# write to target
fl[:2].to_target("file", "_my.grib", encoder=encoder,  metadata={"bitsPerValue": 8})

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls(keys=["time.valid_datetime", "time.base_datetime", "time.step", "metadata.bitsPerValue"])

The code below is equivalent to the one in the previous cell.

In [ ]:
# create an encoder
encoder = ekd.create_encoder("grib", metadata={"date": 20250108})

# write to target
with ekd.create_target("file", "_res_t.grib") as target:
    target.write(fl[:2], encoder=encoder,  metadata={"bitsPerValue": 8})

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls(keys=["time.valid_datetime", "time.base_datetime", "time.step", "metadata.bitsPerValue"])